In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import norm
import statistics
import math
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
import statistics
plt.rcParams["figure.max_open_warning"] = 50

In [8]:
file_Name = []

for x in os.listdir("stock csv"):
    if x.endswith(".csv"):
        file_Name.append(x)

df = pd.DataFrame()

for x in file_Name:
    df2 = pd.read_csv(os.path.join("stock csv", x))
    df2["Name"] = x.replace(".csv","")
    df = pd.concat([df, df2], ignore_index=True)

In [9]:
def text_to_num(text):
    d = {'K': 1_000, 'M': 1_000_000, 'B': 1_000_000_000, 'T': 1_000_000_000_000}
    
    if not isinstance(text, str):
        return text

    text = text.upper().strip()

    # Remove commas first
    text = text.replace(",", "")

    multiplier = 1
    for suffix, value in d.items():
        if suffix in text:
            multiplier = value
            text = text.replace(suffix, '')
            break

    try:
        return float(text) * multiplier
    except ValueError:
        return None

df.dropna(inplace=True)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Open"] = pd.to_numeric(df["Open"], errors="coerce")
df["High"] = pd.to_numeric(df["High"], errors="coerce")
df["Change %"] = (
    df["Change %"]
        .astype(str)
        .str.replace("%", "", regex=False)
)

df["Change %"] = pd.to_numeric(df["Change %"], errors="coerce")
df["Vol."] = df["Vol."].apply(text_to_num)
df["Vol."] =  pd.to_numeric(df["Vol."],errors="coerce")
df = df.sort_values(by="Date").reset_index(drop=True)
df = df.dropna()
df.to_csv("main.csv",index=False)
print("Totle File :", len(file_Name))

Totle File : 30


In [10]:
import pandas as pd
import statistics

cols = ["Price", "High", "Low", "Open"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

main_price_mean = {}
main_price_stdev = {}
data = []
for file in file_Name:

    name = file.replace(".csv", "")

    df2 = df[df["Name"] == name].copy()
    if df2.empty:
        continue

    df2["Date"] = pd.to_datetime(df2["Date"])

    maxyear = df2["Date"].dt.year.max()
    minyear = df2["Date"].dt.year.min()

    print(f"*************** {name} {minyear} → {maxyear} ***************")

    price_mean = {}
    price_stdev = {}

    # -------- YEARLY AGGREGATION --------
    for year in range(minyear, maxyear + 1):

        yer = year + 1

        filtered_df = df2[df2["Date"] < f"{yer}-01-01"].copy()

        if len(filtered_df) < 2:
            print("Not enough data for statistics")
            continue

        # Statistics
        price_mean[yer] = round(filtered_df["Price"].mean(), 2)
        price_stdev[yer] = round(statistics.stdev(filtered_df["Price"]), 2)

     
        if len(filtered_df["Price"]) > 2:
           pass

    # -------- STORE RESULTS --------
    main_price_mean[name] = price_mean
    main_price_stdev[name] = price_stdev

    # -------- BUILD ANALYSIS DATAFRAME --------
    df3 = pd.DataFrame({
        "Year": list(price_mean.keys()),
        "Mean": list(price_mean.values()),
        "Volatility": list(price_stdev.values())
    }).sort_values("Year").reset_index(drop=True)

    if df3.empty or len(df3) < 2:
        continue

    # -------- LOWEST YEAR --------
    lowest_row = df3.loc[df3["Mean"].idxmin()]
    lowest_year = lowest_row["Year"]
    lowest_value = lowest_row["Mean"]

    # -------- YOY GROWTH --------
    df3["YoY_Growth_%"] = df3["Mean"].pct_change() * 100

    yoy_valid = df3.dropna(subset=["YoY_Growth_%"])

    biggest_growth = yoy_valid.loc[yoy_valid["YoY_Growth_%"].idxmax()]
    biggest_drop = yoy_valid.loc[yoy_valid["YoY_Growth_%"].idxmin()]

    # -------- TREND DETECTION --------
    growth_start = None
    sustained_trend = None

    for i in range(1, len(df3) - 1):
        if df3.loc[i, "Mean"] > df3.loc[i-1, "Mean"] and \
           df3.loc[i+1, "Mean"] > df3.loc[i, "Mean"]:
            growth_start = df3.loc[i, "Year"]
            break

    for i in range(2, len(df3)):
        if (df3.loc[i, "Mean"] > df3.loc[i-1, "Mean"] and
            df3.loc[i-1, "Mean"] > df3.loc[i-2, "Mean"]):
            sustained_trend = df3.loc[i-2, "Year"]
            break

    # -------- VOLATILITY --------
    most_volatile = df3.loc[df3["Volatility"].idxmax()]

    regime_change = None
    for i in range(1, len(df3)):
        if df3.loc[i, "Volatility"] > 3 * df3.loc[i-1, "Volatility"]:
            regime_change = df3.loc[i, "Year"]
            break

    # -------- CAGR --------
    start_value = df3.iloc[0]["Mean"]
    end_value = df3.iloc[-1]["Mean"]
    years_count = len(df3) - 1

    if start_value > 0 and years_count > 0:
        cagr = ((end_value / start_value) ** (1 / years_count) - 1) * 100
    else:
        cagr = 0

    # -------- BULL & BEAR RUN --------
    bull_run = bear_run = 0
    max_bull = max_bear = 0

    for i in range(1, len(df3)):
        if df3.loc[i, "Mean"] > df3.loc[i-1, "Mean"]:
            bull_run += 1
            bear_run = 0
        elif df3.loc[i, "Mean"] < df3.loc[i-1, "Mean"]:
            bear_run += 1
            bull_run = 0

        max_bull = max(max_bull, bull_run)
        max_bear = max(max_bear, bear_run)

    # -------- PRINT RESULTS --------
    data.append({
        
        "stock": name,
       "lowest_year": lowest_year,
        "cagr": cagr,
        "bull_run": bull_run,
        "bear_run": max_bear,
        "volatility_year": regime_change
    })
    print(f"===== MARKET ANALYSIS RESULTS {name} =====")
    print(f"Lowest Year: {lowest_year} (Mean={lowest_value})")
    print(f"Growth Started: {growth_start}")
    print(f"Sustained Uptrend Started: {sustained_trend}")
    print(f"Biggest Growth Year: {int(biggest_growth['Year'])} ({biggest_growth['YoY_Growth_%']:.2f}%)")
    print(f"Biggest Drop Year: {int(biggest_drop['Year'])} ({biggest_drop['YoY_Growth_%']:.2f}%)")
    print(f"Most Volatile Year: {int(most_volatile['Year'])}")
    print(f"Volatility Regime Change: {regime_change}")
    print(f"CAGR: {cagr:.2f}%")
    print(f"Longest Bull Run: {max_bull} years")
    print(f"Longest Bear Run: {max_bear} years")
    plt.figure(figsize=(8,5))
    plt.plot(df3["Year"], df3["Mean"], label="Mean Price")
    plt.plot(df3["Year"], df3["Volatility"], label="Volatility (Std Dev)")
    plt.title(f"{name} Mean Price and Volatility by Year")
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.grid(True)
    plt.legend()
    plt.savefig(f"image/{name}.png", dpi=300, bbox_inches='tight')
    plt.close() 
    print(f"\n\n")
    
    

*************** ADEL Historical Data 2000 → 2020 ***************
===== MARKET ANALYSIS RESULTS ADEL Historical Data =====
Lowest Year: 2004.0 (Mean=1.89)
Growth Started: 2005
Sustained Uptrend Started: 2004
Biggest Growth Year: 2008 (80.61%)
Biggest Drop Year: 2002 (-28.39%)
Most Volatile Year: 2021
Volatility Regime Change: 2008
CAGR: 12.29%
Longest Bull Run: 17 years
Longest Bear Run: 3 years



*************** APSE Historical Data 2008 → 2023 ***************
===== MARKET ANALYSIS RESULTS APSE Historical Data =====
Lowest Year: 2010.0 (Mean=107.8)
Growth Started: 2011
Sustained Uptrend Started: 2010
Biggest Growth Year: 2016 (14.76%)
Biggest Drop Year: 2010 (-8.54%)
Most Volatile Year: 2024
Volatility Regime Change: None
CAGR: 7.05%
Longest Bull Run: 14 years
Longest Bear Run: 1 years



*************** AXBK Historical Data 2003 → 2023 ***************
===== MARKET ANALYSIS RESULTS AXBK Historical Data =====
Lowest Year: 2004.0 (Mean=27.31)
Growth Started: 2005
Sustained Uptrend Start

In [11]:
import pandas as pd
import numpy as np

df4 = pd.DataFrame(data).copy()

# ------------------------------
# Required Columns Validation
# ------------------------------
required_cols = ["cagr", "bull_run", "bear_run", "volatility_year"]

missing_cols = [col for col in required_cols if col not in df4.columns]

if missing_cols:
    print("Warning: Missing columns ->", missing_cols)

# ------------------------------
# Safe Numerical Normalization
# ------------------------------
def safe_divide(series, denominator):
    denominator = max(denominator, 1e-9)
    return series / denominator


# Fill NaN values (quant finance best practice)
df4[["cagr","bull_run","bear_run","volatility_year"]] = \
df4[["cagr","bull_run","bear_run","volatility_year"]].fillna(0)


# ------------------------------
# Feature Engineering
# ------------------------------

# CAGR Score (scale 0 → 1)
df4["cagr_score"] = safe_divide(
    df4["cagr"] - df4["cagr"].min(),
    df4["cagr"].max() - df4["cagr"].min()
)

# Bull run strength score
df4["bull_score"] = safe_divide(
    df4["bull_run"] - df4["bull_run"].min(),
    df4["bull_run"].max() - df4["bull_run"].min()
)

# Bear risk penalty (inverse scoring)
df4["bear_score"] = 1 - safe_divide(
    df4["bear_run"] - df4["bear_run"].min(),
    df4["bear_run"].max() - df4["bear_run"].min()
)

# Volatility decay penalty
CURRENT_YEAR = 2025
MAX_VOL_WINDOW = 30

df4["volatility_score"] = 1 - np.clip(
    (CURRENT_YEAR - df4["volatility_year"]) / MAX_VOL_WINDOW,
    0,
    1
)

# ------------------------------
# Investment Score (Weighted Model)
# ------------------------------
# Weight tuning (quant-friendly conservative model)

W_CAGR = 0.5
W_BULL = 0.2
W_BEAR = 0.2
W_VOL = 0.1

df4["investment_score"] = (
    W_CAGR * df4["cagr_score"] +
    W_BULL * df4["bull_score"] +
    W_BEAR * df4["bear_score"] +
    W_VOL * df4["volatility_score"]
)

# ------------------------------
# Ranking
# ------------------------------
df_ranked = df4.sort_values(
    "investment_score",
    ascending=False
).reset_index(drop=True)

print("\n===== Quality Long-Term Compounders =====")
print(df_ranked[["stock","investment_score","cagr","bull_run","bear_run"]])


===== Quality Long-Term Compounders =====
                   stock  investment_score       cagr  bull_run  bear_run
0   BJFN Historical Data          0.891667  29.148968        18         1
1   KTKM Historical Data          0.778587  24.018230        19         1
2   BRTI Historical Data          0.711210  16.158440        20         0
3    SUN Historical Data          0.699896  17.267943        20         0
4   AXBK Historical Data          0.677994  13.988376        20         0
5   BAJE Historical Data          0.663734  17.198925        19         1
6   JSTL Historical Data          0.649006  11.673490        20         0
7   TITN Historical Data          0.644030  16.128009        11         0
8    SBI Historical Data          0.628682  11.258409        20         0
9   RELI Historical Data          0.592839  14.671870         6         0
10  BJFS Historical Data          0.587027  16.800713        12         1
11  LART Historical Data          0.559732  14.883245        15      

In [12]:
df_ranked[["stock","investment_score","cagr","bull_run","bear_run"]].reset_index().to_csv("BEST STOCK RANKING.csv")